# PS-Q1 — Variational Quantum Classifier: Processor A vs B
**Qiskit Fall Fest 2026 · Phase 1 · SRM University-AP**

End-to-end reproducible experiment: data → VQC → co-designed processors → transpilation metrics → noisy evaluation → noisy-in-the-loop training. Run all cells top to bottom (≈ 10–15 min on a laptop CPU).

In [1]:
import sklearn  # import first (avoids an OpenMP loader clash on some Linux setups)
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.problem import *
from src.processors import *
from src.routing import *
from src.metrics import *
from src.visualization import *
SEED = 42; np.random.seed(SEED)
FIG, TAB = 'results/figures', 'results/tables'

## 1. Data
200 samples, 4 features, binary, 80/20 split (stratified, seed 42). Features are min–max scaled to [0, π] using **training statistics only**.

> The guide's call `make_classification(n_samples=200, n_features=4, n_informative=3, random_state=42)` raises `ValueError` with sklearn's default `n_redundant=2`; we pass `n_redundant=0`.

In [2]:
Xtr, Xte, ytr, yte = load_data(save_dir='data')
print(Xtr.shape, Xte.shape, 'train class balance:', ytr.mean(), 'test:', yte.mean())

(160, 4) (40, 4) train class balance: 0.5 test: 0.5


## 2. Processor A & B (designed from scratch)
See `src/processors.py` for the full rationale. Both share basis `{rz, sx, x, cx}`, opt-level 1, seed 42, 4096 shots, and **identical** noise (1q 0.1 %, CX 0.5 %, readout 1 %).

In [3]:
for k in 'AB': save_processor_json(k, f'processors/processor_{k}.json')
procs = {k: load_processor(f'processors/processor_{k}.json') for k in 'AB'}
plot_coupling_pair(procs, f'{FIG}/coupling_maps.png')
for k,p in procs.items(): print(p['name'], '|', len(p['coupling_map']), 'edges |', p['rationale'])
plt.figure(figsize=(10,4)); plt.imshow(plt.imread(f'{FIG}/coupling_maps.png')); plt.axis('off'); plt.show()

Processor A (T-tree, 5q) | 4 edges | Sparse acyclic baseline; no 4-cycle so ring CX must be routed.
Processor B (ring-core + bridges + tail, 9q) | 10 edges | Native 4-cycle for the ring ansatz, second cycle via bridges, low-degree tail.


## 3. The VQC

In [4]:
circ, xp, wp = build_vqc()
print(circ.draw(fold=120, output='text'))
print('trainable weights:', n_weights())

     ┌──────────┐┌──────────┐┌──────────┐                       ┌───┐ ┌──────────┐ ┌──────────┐                       »
q_0: ┤ Ry(x[0]) ├┤ Ry(w[0]) ├┤ Rz(w[1]) ├──■────────────────────┤ X ├─┤ Ry(w[8]) ├─┤ Rz(w[9]) ├──■────────────────────»
     ├──────────┤├──────────┤├──────────┤┌─┴─┐     ┌───────────┐└─┬─┘┌┴──────────┤ └──────────┘┌─┴─┐     ┌───────────┐»
q_1: ┤ Ry(x[1]) ├┤ Ry(w[2]) ├┤ Rz(w[3]) ├┤ X ├──■──┤ Ry(w[10]) ├──┼──┤ Rz(w[11]) ├─────────────┤ X ├──■──┤ Ry(w[17]) ├»
     ├──────────┤├──────────┤├──────────┤└───┘┌─┴─┐└───────────┘  │  ├───────────┤┌───────────┐└───┘┌─┴─┐└───────────┘»
q_2: ┤ Ry(x[2]) ├┤ Ry(w[4]) ├┤ Rz(w[5]) ├─────┤ X ├──────■────────┼──┤ Ry(w[12]) ├┤ Rz(w[13]) ├─────┤ X ├──────■──────»
     ├──────────┤├──────────┤├──────────┤     └───┘    ┌─┴─┐      │  ├───────────┤├───────────┤     └───┘    ┌─┴─┐    »
q_3: ┤ Ry(x[3]) ├┤ Ry(w[6]) ├┤ Rz(w[7]) ├──────────────┤ X ├──────■──┤ Ry(w[14]) ├┤ Rz(w[15]) ├──────────────┤ X ├────»
     └──────────┘└──────────┘└──────────

## 4. Train on the ideal simulator (exact statevector, full batch, SPSA)
Trained once with exact simulation; the same weights are then deployed on A and B so differences come only from hardware constraints.

In [5]:
rng = np.random.default_rng(SEED); w0 = rng.uniform(-np.pi, np.pi, n_weights())
loss_ideal = lambda wv, idx: bce(ideal_p1(circ, xp, wp, Xtr if idx is None else Xtr[idx], wv),
                                 ytr if idx is None else ytr[idx])
w_star, hist_ideal = spsa(loss_ideal, w0, iters=100)
p_tr, p_te = ideal_p1(circ, xp, wp, Xtr, w_star), ideal_p1(circ, xp, wp, Xte, w_star)
ideal_res = dict(train_acc=accuracy(p_tr, ytr), test_acc=accuracy(p_te, yte), test_loss=bce(p_te, yte))
np.save('results/tables/w_star_ideal.npy', w_star)
print(ideal_res)

{'train_acc': 0.86875, 'test_acc': 0.925, 'test_loss': 0.5236781995600426}


### 4b. Classical reference baseline
Same split and scaling. Logistic regression and an RBF-SVM give the classical accuracy ceiling for this dataset, so the VQC can be judged against both an exact quantum reference (ideal simulation) and a classical one.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
base = {}
for name, mdl in [('LogisticRegression', LogisticRegression(max_iter=1000)), ('SVM-RBF', SVC(kernel='rbf', random_state=SEED))]:
    mdl.fit(Xtr, ytr); base[name] = dict(train_acc=mdl.score(Xtr, ytr), test_acc=mdl.score(Xte, yte))
base['VQC (ideal, exact)'] = dict(train_acc=ideal_res['train_acc'], test_acc=ideal_res['test_acc'])
df_base = pd.DataFrame(base).T; display(df_base); df_base.to_csv(f'{TAB}/classical_baseline.csv')

,train_acc,test_acc
LogisticRegression,0.92500,0.875
SVM-RBF,0.95000,0.850
"VQC (ideal, exact)",0.86875,0.925


## 5. Transpile on A vs B — depth, CX, SWAP overhead

In [7]:
mcirc, mxp, mwp = build_vqc(measure=True)
rows = {}
tc = {}
for k,p in procs.items():
    tc[k] = transpile_for(mcirc, p)
    rows[k] = {**circuit_metrics(tc[k]), **swap_overhead(mcirc, p), 'physical_qubits': p['n_qubits'], 'edges': len(p['coupling_map'])}
ref = circuit_metrics(transpile_logical(mcirc)); ref.update(swaps=0, cx_logical=ref['cx'], cx_overhead_from_swaps=0)
rows['all-to-all (ref)'] = ref
df_circ = pd.DataFrame(rows).T; display(df_circ)
df_circ.to_csv(f'{TAB}/circuit_metrics_A_vs_B.csv')
plot_ab_bars(df_circ.loc[['A','B']], ['depth','cx','swaps'], f'{FIG}/circuit_metrics_A_vs_B.png', 'Compiled VQC: Processor A vs B')

,depth,size,cx,one_qubit,swaps,cx_logical,cx_overhead_from_swaps,physical_qubits,edges
A,43.0,85.0,17.0,64.0,3.0,8.0,9.0,5.0,4.0
B,25.0,76.0,8.0,64.0,0.0,8.0,0.0,9.0,10.0
all-to-all (ref),25.0,76.0,8.0,64.0,0.0,8.0,0.0,NaN,NaN


### 5b. Ablation — how much does the topology choice matter?
Same circuit, same seed, five topologies (incl. the guide's reference linear chain and heavy-hex).

In [8]:
abl = {}
for k,p in ABLATION.items():
    t = transpile_for(mcirc, p); abl[k] = {**circuit_metrics(t), 'swaps': swap_overhead(mcirc, p)['swaps'], 'edges': len(p['coupling_map'])}
df_abl = pd.DataFrame(abl).T; display(df_abl); df_abl.to_csv(f'{TAB}/topology_ablation.csv')
plot_bars(df_abl, ['depth','cx','swaps'], f'{FIG}/topology_ablation.png', 'Topology ablation (same VQC)')

,depth,size,cx,one_qubit,swaps,edges
Linear-5 (ref. A),44,92,24,64,6,4
T-tree-5 (A),43,85,17,64,3,4
Ring-5,31,82,14,64,2,5
Heavy-hex-9 (ref. B),25,76,8,64,0,12
Ring-core-9 (B),25,76,8,64,0,10


## 6. Noisy evaluation of the ideal-trained weights (4096 shots)
Synthetic noise identical on both processors. We report accuracy, BCE loss, mean |ΔP(class 1)| vs ideal, and mean Hellinger fidelity between the ideal and noisy 4-bit output distributions.

In [9]:
sim = make_sim(noisy=True)
ideal_dists = [ideal_distribution(circ, xp, wp, xi, w_star) for xi in Xte]
ev = {'ideal': dict(accuracy=ideal_res['test_acc'], bce=ideal_res['test_loss'], mean_abs_dP1=0.0, hellinger_fidelity=1.0)}
for k in 'AB':
    cnts = run_counts(tc[k], mxp, mwp, Xte, w_star, sim)
    p1 = np.array([counts_p1(c) for c in cnts])
    ev[k] = dict(accuracy=accuracy(p1, yte), bce=bce(p1, yte), mean_abs_dP1=float(np.mean(np.abs(p1 - p_te))),
                 hellinger_fidelity=mean_hellinger(ideal_dists, cnts))
df_ev = pd.DataFrame(ev).T; display(df_ev); df_ev.to_csv(f'{TAB}/noisy_eval_A_vs_B.csv')
plot_ab_bars(df_ev, ['accuracy','bce','hellinger_fidelity'], f'{FIG}/noisy_eval_A_vs_B.png', 'Test set under synthetic noise')

,accuracy,bce,mean_abs_dP1,hellinger_fidelity
ideal,0.925,0.523678,0.000000,1.000000
A,0.900,0.556292,0.023656,0.991239
B,0.900,0.546020,0.016631,0.993972


In [10]:
# Bootstrap 95% CIs on noisy test accuracy (n=40 -> wide intervals; read A-vs-B differences accordingly)
def boot_ci(p1, y, n=5000, seed=SEED):
    r = np.random.default_rng(seed); c = ((p1 > 0.5).astype(int) == y).astype(float)
    m = [r.choice(c, size=len(c)).mean() for _ in range(n)]; return np.percentile(m, [2.5, 97.5])
ci_rows = {}
for k in 'AB':
    cn = run_counts(tc[k], mxp, mwp, Xte, w_star, sim); p1 = np.array([counts_p1(c) for c in cn])
    lo, hi = boot_ci(p1, yte); ci_rows[k] = dict(accuracy=accuracy(p1, yte), ci95_low=lo, ci95_high=hi)
df_ci = pd.DataFrame(ci_rows).T; display(df_ci); df_ci.to_csv(f'{TAB}/noisy_accuracy_ci.csv')

,accuracy,ci95_low,ci95_high
A,0.9,0.8,0.975
B,0.9,0.8,0.975


## 7. Noisy-in-the-loop training (convergence on A vs B)
SPSA with the noisy Aer simulator inside the loss, mini-batch 16, 1024 shots, 40 iterations, identical initial weights and seed on both processors — so any convergence difference is due to routing + noise.

In [11]:
sim_t = make_sim(noisy=True); hist_noisy, w_noisy = {}, {}
def make_loss(key):
    def f(wv, idx):
        cn = run_counts(tc[key], mxp, mwp, Xtr[idx], wv, sim_t, shots=1024, seed=SEED)
        return bce(np.array([counts_p1(c) for c in cn]), ytr[idx])
    return f
for k in 'AB':
    w_noisy[k], hist_noisy[k] = spsa(make_loss(k), w0, iters=40, batch=16, n_train=len(Xtr), seed=SEED)
    print(k, 'first-5 mean loss', np.mean(hist_noisy[k][:5]).round(3), '| last-5 mean loss', np.mean(hist_noisy[k][-5:]).round(3))
sm = lambda h, n=5: np.convolve(h, np.ones(n)/n, mode='valid')
plot_curves({'Processor A (noisy)': (sm(hist_noisy['A']), C['A']), 'Processor B (noisy)': (sm(hist_noisy['B']), C['B']),
             'Ideal (exact)': (sm(hist_ideal[:40]), C['ideal'])}, f'{FIG}/training_convergence.png',
            'Training convergence (5-step moving avg of BCE)')
rows = {}
for k in 'AB':
    cn = run_counts(tc[k], mxp, mwp, Xte, w_noisy[k], make_sim(True))
    p1 = np.array([counts_p1(c) for c in cn]); rows[k] = dict(test_acc=accuracy(p1, yte), test_bce=bce(p1, yte),
        loss_first5=float(np.mean(hist_noisy[k][:5])), loss_last5=float(np.mean(hist_noisy[k][-5:])))
df_tr = pd.DataFrame(rows).T; display(df_tr); df_tr.to_csv(f'{TAB}/noisy_training_A_vs_B.csv')
plt.figure(figsize=(6.5,4)); plt.imshow(plt.imread(f'{FIG}/training_convergence.png')); plt.axis('off'); plt.show()

A first-5 mean loss 0.713 | last-5 mean loss 0.639


B first-5 mean loss 0.716 | last-5 mean loss 0.641


,test_acc,test_bce,loss_first5,loss_last5
A,0.850,0.604360,0.712666,0.639107
B,0.875,0.602866,0.715763,0.641169


## 8. Summary table

In [12]:
summary = pd.concat([df_circ.loc[['A','B'], ['depth','cx','swaps']],
                     df_ev.loc[['A','B'], ['accuracy','bce','hellinger_fidelity']].rename(columns={'accuracy':'noisy_test_acc'}),
                     df_tr[['test_acc']].rename(columns={'test_acc':'noisy_trained_test_acc'})], axis=1)
summary['ideal_test_acc'] = ideal_res['test_acc']
display(summary); summary.to_csv(f'{TAB}/summary_A_vs_B.csv'); summary.to_json(f'{TAB}/summary_A_vs_B.json', indent=2)

,depth,cx,swaps,noisy_test_acc,bce,hellinger_fidelity,noisy_trained_test_acc,ideal_test_acc
A,43.0,17.0,3.0,0.9,0.556292,0.991239,0.850,0.925
B,25.0,8.0,0.0,0.9,0.546020,0.993972,0.875,0.925
